# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [ ]:
# importar librerías
import pandas as pd
import numpy as np

In [ ]:

# cargar archivos

orders = pd.read_csv(
    'https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv'
)

catalog = pd.read_csv(
    'https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv'
)

marketing = pd.read_csv(
    'https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv'
)

In [ ]:
# explorar datasets

print("ORDERS")
display(orders.head())
print("Dimensiones:", orders.shape)
orders.info()

print("\n" + "="*60 + "\n")

print("CATALOG")
display(catalog.head())
print("Dimensiones:", catalog.shape)
catalog.info()

print("\n" + "="*60 + "\n")

print("MARKETING")
display(marketing.head())
print("Dimensiones:", marketing.shape)
marketing.info()

In [ ]:
# diagnóstico de calidad de datos

# 1. Valores nulos
print("=== VALORES NULOS ===")

print("\nORDERS")
display(orders.isna().sum().to_frame("nulos"))

print("\nCATALOG")
display(catalog.isna().sum().to_frame("nulos"))

print("\nMARKETING")
display(marketing.isna().sum().to_frame("nulos"))


# 2. Duplicados
print("\n=== DUPLICADOS ===")
print("Orders:", orders.duplicated().sum())
print("Catalog:", catalog.duplicated().sum())
print("Marketing:", marketing.duplicated().sum())


# 3. Valores categóricos relevantes
print("\n=== CATEGORÍAS EN ORDERS ===")

for col in [
    "pais",
    "dispositivo",
    "fuente_referencia",
    "categoria_producto"
]:
    print(f"\n{col}:")
    print(orders[col].value_counts(dropna=False))


print("\n=== CATEGORÍAS EN CATALOG ===")
print(catalog["categoria_producto"].value_counts(dropna=False))


print("\n=== CATEGORÍAS EN MARKETING ===")

for col in ["pais", "canal"]:
    print(f"\n{col}:")
    print(marketing[col].value_counts(dropna=False))


# 4. Revisión de variables numéricas
print("\n=== VARIABLES NUMÉRICAS DE ORDERS ===")
display(
    orders[
        ["cantidad", "precio_unitario", "monto_descuento", "monto_total"]
    ].describe()
)

print("\n=== COSTOS DEL CATÁLOGO ===")
display(catalog["costo_unitario"].describe())

print("\n=== GASTO DE MARKETING ===")
display(marketing["gasto"].describe())

In [ ]:
# investigar anomalías antes de limpiar

# 1. Cantidades inválidas o extremas
print("=== CANTIDADES <= 0 ===")
display(
    orders.loc[
        orders["cantidad"] <= 0,
        [
            "id_pedido", "id_usuario", "nombre_producto",
            "cantidad", "precio_unitario",
            "monto_descuento", "monto_total"
        ]
    ]
)

print("\n=== CANTIDADES MAYORES A 10 ===")
display(
    orders.loc[
        orders["cantidad"] > 10,
        [
            "id_pedido", "id_usuario", "nombre_producto",
            "cantidad", "precio_unitario",
            "monto_descuento", "monto_total"
        ]
    ].sort_values("cantidad", ascending=False)
)


# 2. Totales negativos
print("\n=== MONTOS TOTALES NEGATIVOS ===")
display(
    orders.loc[
        orders["monto_total"] < 0,
        [
            "id_pedido", "nombre_producto",
            "cantidad", "precio_unitario",
            "monto_descuento", "monto_total"
        ]
    ]
)


# 3. Revisar coherencia de la fórmula del pedido
orders_revision = orders.copy()

orders_revision["total_calculado"] = (
    orders_revision["cantidad"] *
    orders_revision["precio_unitario"] -
    orders_revision["monto_descuento"]
)

orders_revision["diferencia_total"] = (
    orders_revision["monto_total"] -
    orders_revision["total_calculado"]
)

print("\n=== DIFERENCIAS ENTRE MONTO TOTAL Y TOTAL CALCULADO ===")
display(
    orders_revision.loc[
        orders_revision["diferencia_total"].abs() > 0.02,
        [
            "id_pedido", "cantidad", "precio_unitario",
            "monto_descuento", "monto_total",
            "total_calculado", "diferencia_total"
        ]
    ].head(30)
)

print(
    "Filas con diferencia:",
    (orders_revision["diferencia_total"].abs() > 0.02).sum()
)


# 4. Revisar los nulos de canal en marketing
print("\n=== MARKETING CON CANAL NULO ===")
display(marketing[marketing["canal"].isna()].head(20))

print("\nCampañas asociadas a canal nulo:")
print(
    marketing.loc[
        marketing["canal"].isna(),
        "id_campaña"
    ].value_counts().head(20)
)


# 5. Revisar duplicados exactos en orders
print("\n=== EJEMPLO DE DUPLICADOS EN ORDERS ===")
display(
    orders[
        orders.duplicated(keep=False)
    ].sort_values("id_pedido").head(20)
)

---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [ ]:
# tu código aquí
# limpieza del dataset orders

orders_clean = orders.copy()

# 1. Convertir fecha al tipo correcto
orders_clean["fecha_hora_pedido"] = pd.to_datetime(
    orders_clean["fecha_hora_pedido"],
    errors="coerce"
)

# 2. Eliminar duplicados exactos
orders_clean = orders_clean.drop_duplicates().copy()

# 3. Estandarizar países
orders_clean["pais"] = (
    orders_clean["pais"]
    .str.strip()
    .str.capitalize()
)

# 4. Estandarizar categoría de producto
orders_clean["categoria_producto"] = (
    orders_clean["categoria_producto"]
    .replace({"Electronica": "Electrónica"})
)

# 5. Eliminar cantidades inválidas o extremadamente anómalas
orders_clean = orders_clean[
    (orders_clean["cantidad"].isna()) |
    ((orders_clean["cantidad"] > 0) &
     (orders_clean["cantidad"] <= 10))
].copy()

print("Filas originales:", len(orders))
print("Filas después de limpieza inicial:", len(orders_clean))

print("\nDuplicados restantes:")
print(orders_clean.duplicated().sum())

print("\nPaíses:")
print(orders_clean["pais"].value_counts(dropna=False))

print("\nCategorías:")
print(orders_clean["categoria_producto"].value_counts(dropna=False))

print("\nCantidad:")
display(orders_clean["cantidad"].describe())

print("\nTipo de fecha:")
print(orders_clean["fecha_hora_pedido"].dtype)

In [ ]:
# revisar si los valores nulos pueden recuperarse

print("=== NULOS ACTUALES EN ORDERS ===")
display(orders_clean.isna().sum().to_frame("nulos"))

# Revisar productos disponibles en el catálogo
print("\n=== PRODUCTOS DEL CATÁLOGO ===")
display(
    catalog[
        ["nombre_producto", "categoria_producto"]
    ].sort_values("nombre_producto")
)

# Registros donde falta categoría pero existe producto
print("\n=== CATEGORÍA NULA CON PRODUCTO DISPONIBLE ===")
display(
    orders_clean.loc[
        orders_clean["categoria_producto"].isna()
        & orders_clean["nombre_producto"].notna(),
        ["nombre_producto", "categoria_producto"]
    ].head(20)
)

# Registros donde falta producto
print("\n=== PRODUCTO NULO ===")
display(
    orders_clean.loc[
        orders_clean["nombre_producto"].isna(),
        [
            "id_pedido",
            "categoria_producto",
            "cantidad",
            "precio_unitario",
            "monto_total"
        ]
    ].head(20)
)

# Revisar filas con variables numéricas faltantes
print("\n=== NULOS NUMÉRICOS ===")
display(
    orders_clean.loc[
        orders_clean[
            ["cantidad", "precio_unitario", "monto_descuento"]
        ].isna().any(axis=1),
        [
            "id_pedido",
            "nombre_producto",
            "cantidad",
            "precio_unitario",
            "monto_descuento",
            "monto_total"
        ]
    ].head(30)
)

In [ ]:
# completar y finalizar limpieza de orders

# mapa producto -> categoría a partir del catálogo
mapa_categoria = catalog.set_index(
    "nombre_producto"
)["categoria_producto"]

# recuperar categorías faltantes usando el nombre del producto
orders_clean["categoria_producto"] = (
    orders_clean["categoria_producto"]
    .fillna(
        orders_clean["nombre_producto"].map(mapa_categoria)
    )
)

# eliminar registros sin información esencial para el análisis
columnas_esenciales = [
    "nombre_producto",
    "categoria_producto",
    "cantidad",
    "precio_unitario",
    "monto_descuento"
]

orders_clean = (
    orders_clean
    .dropna(subset=columnas_esenciales)
    .copy()
)

print("=== VALIDACIÓN FINAL DE ORDERS ===")

print("\nDimensiones:")
print(orders_clean.shape)

print("\nValores nulos:")
display(orders_clean.isna().sum().to_frame("nulos"))

print("\nDuplicados:")
print(orders_clean.duplicated().sum())

print("\nTipos de datos:")
print(orders_clean.dtypes)

print("\nCategorías:")
print(
    orders_clean["categoria_producto"]
    .value_counts(dropna=False)
)

print("\nCantidad:")
display(orders_clean["cantidad"].describe())

In [ ]:
# completar valores categóricos faltantes

orders_clean["pais"] = (
    orders_clean["pais"]
    .fillna("Desconocido")
)

orders_clean["dispositivo"] = (
    orders_clean["dispositivo"]
    .fillna("Desconocido")
)

# convertir cantidad a entero
orders_clean["cantidad"] = orders_clean["cantidad"].astype(int)

print("=== ORDERS CLEAN ===")
print("Dimensiones:", orders_clean.shape)
print("Nulos totales:", orders_clean.isna().sum().sum())
print("Duplicados:", orders_clean.duplicated().sum())

print("\nPaís:")
print(orders_clean["pais"].value_counts())

print("\nDispositivo:")
print(orders_clean["dispositivo"].value_counts())

print("\nTipos:")
print(orders_clean.dtypes)

In [ ]:
# validación y limpieza de catalog

catalog_clean = catalog.copy()

# limpiar espacios en variables categóricas
for col in ["nombre_producto", "categoria_producto", "proveedor"]:
    catalog_clean[col] = catalog_clean[col].str.strip()

print("=== CATALOG CLEAN ===")

print("\nDimensiones:")
print(catalog_clean.shape)

print("\nValores nulos:")
display(catalog_clean.isna().sum().to_frame("nulos"))

print("\nDuplicados:")
print(catalog_clean.duplicated().sum())

print("\nProductos duplicados:")
print(catalog_clean["nombre_producto"].duplicated().sum())

print("\nCostos inválidos (<= 0):")
print((catalog_clean["costo_unitario"] <= 0).sum())

print("\nCategorías:")
print(catalog_clean["categoria_producto"].value_counts(dropna=False))

print("\nResumen de costos:")
display(catalog_clean["costo_unitario"].describe())

display(catalog_clean)

In [ ]:
# limpieza del dataset marketing

marketing_clean = marketing.copy()

# 1. Convertir fecha
marketing_clean["fecha"] = pd.to_datetime(
    marketing_clean["fecha"],
    errors="coerce"
)

# 2. Limpiar espacios en variables categóricas
for col in ["pais", "id_campaña", "canal"]:
    marketing_clean[col] = marketing_clean[col].str.strip()

# 3. Recuperar canales faltantes desde id_campaña
marketing_clean.loc[
    marketing_clean["canal"].isna(),
    "canal"
] = (
    marketing_clean.loc[
        marketing_clean["canal"].isna(),
        "id_campaña"
    ]
    .str.rsplit("_", n=1)
    .str[0]
)

# 4. Validación
print("=== MARKETING CLEAN ===")

print("\nDimensiones:")
print(marketing_clean.shape)

print("\nValores nulos:")
display(marketing_clean.isna().sum().to_frame("nulos"))

print("\nDuplicados:")
print(marketing_clean.duplicated().sum())

print("\nCanales:")
print(marketing_clean["canal"].value_counts(dropna=False))

print("\nPaíses:")
print(marketing_clean["pais"].value_counts(dropna=False))

print("\nGastos inválidos (<= 0):")
print((marketing_clean["gasto"] <= 0).sum())

print("\nTipo de fecha:")
print(marketing_clean["fecha"].dtype)

print("\nResumen del gasto:")
display(marketing_clean["gasto"].describe())

In [ ]:
# limpieza del dataset marketing

marketing_clean = marketing.copy()

# 1. Convertir fecha
marketing_clean["fecha"] = pd.to_datetime(
    marketing_clean["fecha"],
    errors="coerce"
)

# 2. Limpiar espacios en variables categóricas
for col in ["pais", "id_campaña", "canal"]:
    marketing_clean[col] = marketing_clean[col].str.strip()

# 3. Recuperar canales faltantes desde id_campaña
marketing_clean.loc[
    marketing_clean["canal"].isna(),
    "canal"
] = (
    marketing_clean.loc[
        marketing_clean["canal"].isna(),
        "id_campaña"
    ]
    .str.rsplit("_", n=1)
    .str[0]
)

# 4. Validación
print("=== MARKETING CLEAN ===")

print("\nDimensiones:")
print(marketing_clean.shape)

print("\nValores nulos:")
display(marketing_clean.isna().sum().to_frame("nulos"))

print("\nDuplicados:")
print(marketing_clean.duplicated().sum())

print("\nCanales:")
print(marketing_clean["canal"].value_counts(dropna=False))

print("\nPaíses:")
print(marketing_clean["pais"].value_counts(dropna=False))

print("\nGastos inválidos (<= 0):")
print((marketing_clean["gasto"] <= 0).sum())

print("\nTipo de fecha:")
print(marketing_clean["fecha"].dtype)

print("\nResumen del gasto:")
display(marketing_clean["gasto"].describe())

In [ ]:
# validación final de los 3 datasets

print("=== VALIDACIÓN FINAL PASO 1 ===")

print("\nORDERS")
print("Filas:", len(orders_clean))
print("Nulos:", orders_clean.isna().sum().sum())
print("Duplicados:", orders_clean.duplicated().sum())

print("\nCATALOG")
print("Filas:", len(catalog_clean))
print("Nulos:", catalog_clean.isna().sum().sum())
print("Duplicados:", catalog_clean.duplicated().sum())

print("\nMARKETING")
print("Filas:", len(marketing_clean))
print("Nulos:", marketing_clean.isna().sum().sum())
print("Duplicados:", marketing_clean.duplicated().sum())

# validar que todos los productos de orders existan en catalog
productos_sin_catalogo = set(
    orders_clean["nombre_producto"].unique()
) - set(
    catalog_clean["nombre_producto"].unique()
)

print("\nProductos de orders sin correspondencia en catalog:")
print(productos_sin_catalogo)

# validar categorías
print("\nCategorías Orders:")
print(sorted(orders_clean["categoria_producto"].unique()))

print("\nCategorías Catalog:")
print(sorted(catalog_clean["categoria_producto"].unique()))

# validar canales
print("\nCanales Orders:")
print(sorted(orders_clean["fuente_referencia"].unique()))

print("\nCanales Marketing:")
print(sorted(marketing_clean["canal"].unique()))

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [ ]:
# exportar datasets limpios

orders_clean.to_csv('orders_clean.csv', index=False)
catalog_clean.to_csv('catalog_clean.csv', index=False)
marketing_clean.to_csv('marketing_clean.csv', index=False)

print("Archivos exportados correctamente:")
print("- orders_clean.csv")
print("- catalog_clean.csv")
print("- marketing_clean.csv")

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [ ]:

# ==========================================
# PASO 2.1 - CÁLCULO DE KPIs PRINCIPALES
# ==========================================

# unir pedidos con el costo unitario del catálogo
ventas = orders_clean.merge(
    catalog_clean[
        ["nombre_producto", "costo_unitario"]
    ],
    on="nombre_producto",
    how="left"
)

# calcular costo de producto por pedido
ventas["costo_producto"] = (
    ventas["cantidad"] * ventas["costo_unitario"]
)

# --------------------------
# PARTE 1: RENTABILIDAD
# --------------------------

revenue_total = ventas["monto_total"].sum()
costo_productos = ventas["costo_producto"].sum()
gasto_marketing = marketing_clean["gasto"].sum()

costo_total = costo_productos + gasto_marketing
profit = revenue_total - costo_total
margen_profit = (profit / revenue_total) * 100

print("=== RENTABILIDAD DEL NEGOCIO ===")
print(f"Revenue total: ${revenue_total:,.2f}")
print(f"Costo de productos: ${costo_productos:,.2f}")
print(f"Gasto de marketing: ${gasto_marketing:,.2f}")
print(f"Costo total: ${costo_total:,.2f}")
print(f"Profit: ${profit:,.2f}")
print(f"Margen de profit: {margen_profit:.2f}%")

In [ ]:
# --------------------------
# PARTE 2: COMPORTAMIENTO DE VENTAS
# --------------------------

ticket_promedio = ventas["monto_total"].mean()

cantidad_promedio = ventas["cantidad"].mean()

producto_mas_vendido = (
    ventas.groupby("nombre_producto")["cantidad"]
    .sum()
    .sort_values(ascending=False)
)

marketing_por_canal = (
    marketing_clean.groupby("canal")["gasto"]
    .sum()
    .sort_values(ascending=False)
)

print("=== COMPORTAMIENTO DE VENTAS ===")

print(f"\nTicket promedio por orden: ${ticket_promedio:,.2f}")
print(f"Cantidad promedio de productos por orden: {cantidad_promedio:.2f}")

print("\nProducto más vendido:")
display(producto_mas_vendido.head(1).to_frame("unidades_vendidas"))

print("\nGasto de marketing por canal:")
display(marketing_por_canal.to_frame("gasto_marketing"))

In [ ]:
print("=== VALIDACIÓN DEL MERGE ===")
print("Pedidos antes del merge:", len(orders_clean))
print("Pedidos después del merge:", len(ventas))
print("Costos unitarios nulos:", ventas["costo_unitario"].isna().sum())

### Conclusiones de rentabilidad y comportamiento de ventas

El negocio generó ingresos totales por **$9,610,018.94**, frente a un costo total de **$6,700,712.54**, considerando tanto el costo de los productos como la inversión en marketing.

Como resultado, se obtuvo una ganancia (profit) de **$2,909,306.40**, equivalente a un margen de **30.27% sobre los ingresos**. Por lo tanto, durante el periodo analizado el negocio fue rentable.

La inversión total en marketing fue de **$2,871,843.53**, mientras que el costo asociado a los productos vendidos alcanzó **$3,828,869.01**.

En cuanto al comportamiento de compra, el ticket promedio fue de **$385.85 por pedido** y los clientes adquirieron en promedio **1.50 productos por orden**.

El producto con mayor volumen de ventas fue **Vacuum-Pro-Black**, con **6,284 unidades vendidas**.

La inversión de marketing se distribuyó de manera relativamente equilibrada entre los tres canales. **Social** presentó el mayor gasto con **$976,818.37**, seguido de **Organic** con **$972,650.96** y **Paid Search** con **$922,374.20**.

Estos resultados muestran un negocio rentable, aunque el nivel de inversión en marketing es relevante dentro de la estructura de costos. Por ello, el siguiente nivel de análisis debe evaluar qué segmentos, productos y canales generan mayor rentabilidad y no únicamente mayor volumen o inversión.

---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine

# Example only: credentials must be stored as environment variables.
db_config = {
    "user": os.getenv("DB_USER"),
    "pwd": os.getenv("DB_PASSWORD"),
    "host": os.getenv("DB_HOST"),
    "port": os.getenv("DB_PORT", "5432"),
    "db": os.getenv("DB_NAME"),
}

connection_string = "postgresql://{user}:{pwd}@{host}:{port}/{db}".format(**db_config)
engine = create_engine(connection_string, connect_args={"sslmode": "require"})


In [ ]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

In [ ]:
# PARTE 1: Totales del funnel
# ======================

query_totals ='''
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios
FROM events
GROUP BY nombre_evento
ORDER BY usuarios DESC;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

In [ ]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH funnel AS (
    SELECT
        COUNT(DISTINCT CASE WHEN nombre_evento = 'first_visit' THEN id_usuario END) AS first_visit,
        COUNT(DISTINCT CASE WHEN nombre_evento = 'add_to_cart' THEN id_usuario END) AS add_to_cart,
        COUNT(DISTINCT CASE WHEN nombre_evento = 'select_item' THEN id_usuario END) AS select_item,
        COUNT(DISTINCT CASE WHEN nombre_evento = 'begin_checkout' THEN id_usuario END) AS begin_checkout,
        COUNT(DISTINCT CASE WHEN nombre_evento = 'add_payment_info' THEN id_usuario END) AS add_payment_info,
        COUNT(DISTINCT CASE WHEN nombre_evento = 'purchase' THEN id_usuario END) AS purchase
    FROM events
)

SELECT
    ROUND(add_to_cart * 100.0 / first_visit, 2) AS conversion_visit_cart,
    ROUND(select_item * 100.0 / add_to_cart, 2) AS conversion_cart_select,
    ROUND(begin_checkout * 100.0 / select_item, 2) AS conversion_select_checkout,
    ROUND(add_payment_info * 100.0 / begin_checkout, 2) AS conversion_checkout_payment,
    ROUND(purchase * 100.0 / add_payment_info, 2) AS conversion_payment_purchase,
    ROUND(purchase * 100.0 / first_visit, 2) AS conversion_final
FROM funnel;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [ ]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

In [ ]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

In [ ]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohortes AS (
    SELECT
        id_usuario,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS cohorte
    FROM users
),

retencion AS (
    SELECT
        c.cohorte,
        c.id_usuario,
        MAX(CASE
            WHEN ua.dias_despues_registro = 7 AND ua.activo = 1
            THEN 1 ELSE 0
        END) AS retenido_w1,
        MAX(CASE
            WHEN ua.dias_despues_registro = 14 AND ua.activo = 1
            THEN 1 ELSE 0
        END) AS retenido_w2,
        MAX(CASE
            WHEN ua.dias_despues_registro = 21 AND ua.activo = 1
            THEN 1 ELSE 0
        END) AS retenido_w3
    FROM cohortes c
    LEFT JOIN user_activity ua
        ON c.id_usuario = ua.id_usuario
    GROUP BY c.cohorte, c.id_usuario
)

SELECT
    cohorte,
    COUNT(*) AS usuarios_iniciales,
    SUM(retenido_w1) AS retenido_w1,
    SUM(retenido_w2) AS retenido_w2,
    SUM(retenido_w3) AS retenido_w3,
    ROUND(SUM(retenido_w1) * 100.0 / COUNT(*), 2) AS semana_1,
    ROUND(SUM(retenido_w2) * 100.0 / COUNT(*), 2) AS semana_2,
    ROUND(SUM(retenido_w3) * 100.0 / COUNT(*), 2) AS semana_3
FROM retencion
GROUP BY cohorte
ORDER BY cohorte;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** ...
   - **H₁ (Hipótesis alternativa):** ...
   
**Test estadístico:** ...  
**Nivel de significancia alpha:** ...

In [ ]:
import pandas as pd

# Cargar dataset del experimento A/B
experiment = pd.read_csv(
    'https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv'
)

# Convertir timestamp a fecha
experiment['timestamp'] = pd.to_datetime(experiment['timestamp'])

print("=== INFORMACIÓN DEL EXPERIMENTO ===")
print("Dimensiones:", experiment.shape)

print("\n=== PRIMERAS FILAS ===")
display(experiment.head())

print("\n=== VALORES NULOS ===")
display(experiment.isnull().sum().to_frame("nulos"))

print("\n=== DISTRIBUCIÓN POR VARIANTE ===")
display(experiment['variante'].value_counts().to_frame("usuarios"))

print("\n=== CONVERSIÓN POR VARIANTE ===")
display(
    experiment.groupby('variante')['convirtio']
    .agg(['count', 'sum', 'mean'])
    .rename(columns={
        'count': 'usuarios',
        'sum': 'conversiones',
        'mean': 'tasa_conversion'
    })
)



In [ ]:
from statsmodels.stats.proportion import proportions_ztest

# Conversiones de cada grupo
conversiones = [
    experiment.loc[experiment['variante'] == 'control', 'convirtio'].sum(),
    experiment.loc[experiment['variante'] == 'tratamiento', 'convirtio'].sum()
]

# Número de usuarios de cada grupo
usuarios = [
    (experiment['variante'] == 'control').sum(),
    (experiment['variante'] == 'tratamiento').sum()
]

# Test Z de dos proporciones
z_stat, p_value = proportions_ztest(
    count=conversiones,
    nobs=usuarios,
    alternative='two-sided'
)

print("=== TEST A/B DE CONVERSIÓN ===")
print(f"Conversiones control: {conversiones[0]} / {usuarios[0]}")
print(f"Conversiones tratamiento: {conversiones[1]} / {usuarios[1]}")
print(f"Estadístico Z: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"Alpha: {0.05}")

### Interpretación del experimento

El grupo control obtuvo una tasa de conversión de 15.69%, mientras que el grupo tratamiento alcanzó 16.29%, una diferencia de aproximadamente 0.60 puntos porcentuales a favor del tratamiento.

Sin embargo, el test Z de dos proporciones obtuvo un p-value de 0.4161, superior al nivel de significancia establecido (α = 0.05).

Por lo tanto, no se rechaza la hipótesis nula. No existe evidencia estadísticamente significativa para afirmar que el cambio en la interfaz del checkout haya generado una mejora en la tasa de conversión.

### Recomendación

No se recomienda implementar el cambio de UI basándose únicamente en los resultados actuales. Se sugiere continuar evaluando la modificación mediante experimentos adicionales o una muestra mayor antes de tomar una decisión definitiva.

---



## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)

- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc


---